# Benchmarking to BIOSCREEN

This notebook provides a comparison of concentrations calculated with the *mibitrans* package and concentrations calculated with the *Excel*-based distributions *BIOSCREEN*$^{1,2}$ and *BIOSCREEN-AT*$^{3}$. The notebooks benchmarks *mibitrans* showing that equation implementations are correct. 

**Authors:** Jorrit Bakker, Alraune Zech


## Background

*BIOSCREEN-AT*$^{3}$ implements the exact solution of the transport equations for the general setting of the package, following the solution of Wexler, 1992$^4$. Calculations require numerical integration. The model class `Mibitrans` of the *mibitrans* package implements the same exact equation in python for contaminant transport modelling.

*BIOSCREEN*$^{1,2}$ implements an approximate, but fully closed-form, equation as solution of the contaminant transport, based on the empirical model of Domenico, 1987$^5$. West et al., 2007$^6$ discussed in detail the approximations and their effects. The same equation is implemented in the *mibitrans* package as `Bioscreen` model class. Mostly for comparison with BIOSCREEN.

## BIOSCREEN example dataset and BIOSCREEN results

We use for comparison the set of transport and model parameters, those provided as *BIOSCREEN example dataset* in BIOSCREEN. We ran both, BIOSCREEN and BIOSCREEN-AT in Excel to retreive concentrations for comparison. Values are listed below in arrays.


## References

$^1$ Newell, C. J., R. K. McLeod, and J. R. Gonzales (1996), BIOSCREEN Natural Attenuation Decision Support System User’s Manual Version 1.3, Tech. Rep. EPA/600/R-96/087, US EPA

$^2$ Newell, C. J., McLeod, R. K., & Gonzales, J. R. (1997), BIOSCREEN natural attenuation decision support system version 1.4 revisions, Tech. rep., U.S. EPA.

$^3$[Karanovic, M., C. J. Neville, and C. B. Andrews,(2007), BIOSCREEN-AT: BIOSCREEN with an exact analytical solution, Groundwater, 45 (2), 242–245.](https://doi.org/doi:10.1111/j.1745-6584.2006.00296.x)

$^4$ Wexler, E. J., (1992),  Analytical solutions for one-, two-, and three-dimensional solute transport in ground-water systems with uniform flow, Tech. Rep. 03-B7, U.S. G.P.O.

$^5$ [Domenico, P. A., 1987, An analytical model for multidimensional transport of a decaying contaminant
species, Journal of Hydrology, 91 (1), 49–58.](https://doi.org/10.1016/0022-1694(87)90127-2)

$^6$ [West, M. R., B. H. Kueper, and M. J. Ungs, (2007), On the use and error of approximation in the Domenico (1987) solution, Groundwater, 45 (2), 126–135.](https://doi.org/10.1111/j.1745-6584.2006.00280.x)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mibitrans as mbt

In [ ]:
ft = 3.281 # factor to convert ft to m

## Benchmarking

### BIOSCREEN example dataset

In [ ]:
bio_hydro = mbt.HydrologicalParameters(
    velocity=1609.1 / ft / 365, #[m/d]
    porosity=0.25, #[-]
    alpha_x=28.5 / ft, #[m]
    alpha_y=2.9 / ft, #[m]
    alpha_z=0 / ft, #[m]
)

bio_att = mbt.AttenuationParameters(
    bulk_density=1700, #[kg/m3]
    partition_coefficient=0.038, #[m3/kg]
    fraction_organic_carbon=0.0008, #[-]
    half_life=0, #[d]
)

bio_source = mbt.SourceParameters(
    source_zone_boundary=np.array([50, 75, 125]) / ft, #[m]
    source_zone_concentration=np.array([9, 2.8, 0.07]), #[g/m3]
    depth=10 / ft, #[m]
    total_mass="inf", #[g]
)

bio_model = mbt.ModelParameters(
    model_length=1450 / ft, #[m]
    model_width=300 / ft, #[m]
    model_time=5 * 365, #[d]
    dx=5 / ft, #[m]
    dy=5 / ft, #[m]
    dt=365/2, #[d]
)

bio_ea = mbt.ElectronAcceptors(
    delta_oxygen=5.78, #[g/m3]
    delta_nitrate=17, #[g/m3]
    ferrous_iron=11.3, #[g/m3]
    delta_sulfate=100, #[g/m3]
    methane=0.414, #[g/m3]
)

### Simulation results from BIOSCREEN

In [ ]:
bio_x = np.array([0,145,290,435,580,725,870,1015,1160,1305,1450])/ft
bio_y = np.array([-150,-75,0,75,150])/ft
bio_t = np.array([0.5*365, 365, 2*365, 5*365])

BIOSCREEN_data_nodecay = np.array([
[
    [0.0,0.0273283,0.1468138,0.3000519,0.3614301,0.2760113,0.1312136,0.0378381,0.0064984,0.0006571,0.0000389],
    [2.8,2.6143922,3.0078986,2.9053992,2.2669254,1.2981351,0.5016908,0.1236433,0.0187654,0.0017170,0.0000935],
    [9.0,8.4351708,7.2666113,5.8995953,4.0906094,2.1512427,0.7797674,0.1828237,0.0266626,0.0023614,0.0001251],
    [2.8,2.6143922,3.0078986,2.9053992,2.2669254,1.2981351,0.5016908,0.1236433,0.0187654,0.0017170,0.0000935],
    [0.0,0.0273283,0.1468138,0.3000519,0.3614301,0.2760113,0.1312136,0.0378381,0.0064984,0.0006571,0.0000389]
],
[
    [0.0,0.0274302,0.1508231,0.3397355,0.5363354,0.7117680,0.8424022,0.8955887,0.8385638,0.6707641,0.4448148],
    [2.8,2.6241429,3.0900404,3.2896550,3.3639493,3.3475847,3.2208979,2.9265090,2.4215047,1.7525951,1.0700174],
    [9.0,8.4666308,7.4650529,6.6798508,6.0701612,5.5475480,5.0061731,4.3272481,3.4405678,2.4103587,1.4322773],
    [2.8,2.6241429,3.0900404,3.2896550,3.3639493,3.3475847,3.2208979,2.9265090,2.4215047,1.7525951,1.0700174],
    [0.0,0.0274302,0.1508231,0.3397355,0.5363354,0.7117680,0.8424022,0.8955887,0.8385638,0.6707641,0.4448148]
],
[
    [0.0,0.0274304,0.1508347,0.3399257,0.5380249,0.7216271,0.8833805,1.0224062,1.1403052,1.2394019,1.3217872],
    [2.8,2.6241643,3.0902782,3.2914970,3.3745460,3.3939539,3.3775771,3.3409097,3.2928375,3.2383513,3.1796048],
    [9.0,8.4666997,7.4656275,6.6835911,6.0892827,5.6243902,5.2496961,4.9399969,4.6785913,4.4537315,4.2560766],
    [2.8,2.6241643,3.0902782,3.2914970,3.3745460,3.3939539,3.3775771,3.3409097,3.2928375,3.2383513,3.1796048],
    [0.0,0.0274304,0.1508347,0.3399257,0.5380249,0.7216271,0.8833805,1.0224062,1.1403052,1.2394019,1.3217872]
],
[
    [0.0,0.0274304,0.1508347,0.3399257,0.5380250,0.7216274,0.8833823,1.0224178,1.1403686,1.2396981,1.3229797],
    [2.8,2.6241643,3.0902782,3.2914971,3.3745462,3.3939550,3.3775839,3.3409476,3.2930206,3.2391254,3.1824734],
    [9.0,8.4666997,7.4656275,6.6835912,6.0892830,5.6243920,5.2497068,4.9400530,4.6788515,4.4547961,4.2599163],
    [2.8,2.6241643,3.0902782,3.2914971,3.3745462,3.3939550,3.3775839,3.3409476,3.2930206,3.2391254,3.1824734],
    [0.0,0.0274304,0.1508347,0.3399257,0.5380250,0.7216274,0.8833823,1.0224178,1.1403686,1.2396981,1.3229797]
]
])

BIOSCREEN_data_lineardecay = np.array([
[
    [0.0,0.0140858,0.0396948,0.0451984,0.0340673,0.0186897,0.0071604,0.0017981,0.0002821,0.0000268,0.0000015],
    [2.8,1.3475325,0.8132616,0.4376557,0.2136733,0.0879014,0.0273777,0.0058757,0.0008145,0.0000701,0.0000037],
    [9.0,4.3477281,1.9647125,0.8886873,0.3855680,0.1456684,0.0425526,0.0086880,0.0011572,0.0000964,0.0000049],
    [2.8,1.3475325,0.8132616,0.4376557,0.2136733,0.0879014,0.0273777,0.0058757,0.0008145,0.0000701,0.0000037],
    [0.0,0.0140858,0.0396948,0.0451984,0.0340673,0.0186897,0.0071604,0.0017981,0.0002821,0.0000268,0.0000015]
],
[
    [0.0,0.0140883,0.0397879,0.0460530,0.0374360,0.0257826,0.0161884,0.0095640,0.0053614,0.0028197,0.0013529],
    [2.8,1.3477712,0.8151693,0.4459310,0.2348022,0.1212608,0.0618959,0.0312522,0.0154819,0.0073674,0.0032545],
    [9.0,4.3484984,1.9693211,0.9054908,0.4236947,0.2009509,0.0962034,0.0462107,0.0219972,0.0101325,0.0043564],
    [2.8,1.3477712,0.8151693,0.4459310,0.2348022,0.1212608,0.0618959,0.0312522,0.0154819,0.0073674,0.0032545],
    [0.0,0.0140883,0.0397879,0.0460530,0.0374360,0.0257826,0.0161884,0.0095640,0.0053614,0.0028197,0.0013529]
],
[
    [0.0,0.0140883,0.0397880,0.0460532,0.0374372,0.0257893,0.0162144,0.0096384,0.0055214,0.0030828,0.0016897],
    [2.8,1.3477712,0.8151695,0.4459323,0.2348097,0.1212919,0.0619951,0.0314953,0.0159439,0.0080548,0.0040646],
    [9.0,4.3484984,1.9693216,0.9054935,0.4237081,0.2010025,0.0963576,0.0465702,0.0226538,0.0110778,0.0054407],
    [2.8,1.3477712,0.8151695,0.4459323,0.2348097,0.1212919,0.0619951,0.0314953,0.0159439,0.0080548,0.0040646],
    [0.0,0.0140883,0.0397880,0.0460532,0.0374372,0.0257893,0.0162144,0.0096384,0.0055214,0.0030828,0.0016897]
],
[
    [0.0,0.0140883,0.0397880,0.0460532,0.0374372,0.0257893,0.0162144,0.0096384,0.0055214,0.0030828,0.0016897],
    [2.8,1.3477712,0.8151695,0.4459323,0.2348097,0.1212919,0.0619951,0.0314953,0.0159439,0.0080548,0.0040646],
    [9.0,4.3484984,1.9693216,0.9054935,0.4237081,0.2010025,0.0963576,0.0465702,0.0226538,0.0110778,0.0054407],
    [2.8,1.3477712,0.8151695,0.4459323,0.2348097,0.1212919,0.0619951,0.0314953,0.0159439,0.0080548,0.0040646],
    [0.0,0.0140883,0.0397880,0.0460532,0.0374372,0.0257893,0.0162144,0.0096384,0.0055214,0.0030828,0.0016897]
]
])

BIOSCREEN_data_instant = np.array([
[
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[2.8,1.3824385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[9.0,8.3318798,6.4731581,2.3635198,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[2.8,1.3824385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
],
[
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[2.8,1.4906657,0.0648612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[9.0,8.4660257,7.4046305,6.3311115,5.1648196,3.7566839,1.7238633,0.0,0.0,0.0,0.0],
[2.8,1.4906657,0.0648612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
],
[
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[2.8,1.4909029,0.0669935,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[9.0,8.4663197,7.4073278,6.3501310,5.2681446,4.1915197,3.1520539,2.1673353,1.2435009,0.3783501,0.0],
[2.8,1.4909029,0.0669935,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
],
[
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[2.8,1.4909029,0.0669935,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[9.0,8.4663197,7.4073278,6.3501311,5.2681460,4.1915298,3.1521163,2.1676734,1.2451070,0.3850461,0.0],
[2.8,1.4909029,0.0669935,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
[0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
]
])


### BIOSCREEN-AT example dataset

Input parameters are the default parameters when opening up BIOSCREEN-AT. All imperial units are converted to consistent metric units.


In [ ]:
bat_hydro = mbt.HydrologicalParameters(
    velocity=335.2 / ft / 365, #[m/d]
    porosity=0.25, #[-]
    alpha_x=28.887 / ft, #[m]
    alpha_y=2.889 / ft, #[m]
    alpha_z=0.289 / ft, #[m]
)

bat_att = mbt.AttenuationParameters(
    retardation=1.20672, #[-]
    half_life=0, #[d]
)

bat_source = mbt.SourceParameters(
    source_zone_boundary=np.array([50 / ft]), #[m]
    source_zone_concentration=np.array([9]), #[g/m3]
    depth=10 / ft, #[m]
    total_mass="inf", #[g]
)

bat_model = mbt.ModelParameters(
    model_length=1500 / ft, #[m]
    model_width=300 / ft, #[m]
    model_time=5 * 365, #[d]
    dx=5 / ft, #[m]
    dy=5 / ft, #[m]
    dt=365, #[d]
)

bat_ea = mbt.ElectronAcceptors(
    delta_oxygen=5.78, #[g/m3]
    delta_nitrate=17, #[g/m3]
    ferrous_iron=11.3, #[g/m3]
    delta_sulfate=100, #[g/m3]
    methane=0.414, #[g/m3]
)

### Simulation results from BIOSCREEN-AT

In [ ]:
bat_x = np.array([0, 150, 300, 450, 600, 750, 900, 1050, 1200, 1350, 1500])/ft
bat_y = np.array([-150,-75,0,75,150])/ft
bat_t = np.array([365, 2*365, 3*365, 4*365, 5*365])

BIOSCREENAT_data_nodecay = np.array([
[
    [0,0.003390686,0.006516547,0.002285627,0.000188474,3.85911E-06,1.98085E-08,2.5472E-11,8.18479E-15,6.55517E-19,1.27271E-23],
    [0,1.002253997,0.67934151,0.154322733,0.010496842,0.000195157,9.49058E-07,1.18084E-09,3.71365E-13,2.93065E-17,5.60443E-22],
    [9,5.931852163,2.586809007,0.514064509,0.033217184,0.000603112,2.89617E-06,3.57666E-09,1.11947E-12,8.80575E-17,1.67838E-21],
    [0,1.002253997,0.67934151,0.154322733,0.010496842,0.000195157,9.49058E-07,1.18084E-09,3.71365E-13,2.93065E-17,5.60443E-22],
    [0,0.003390686,0.006516547,0.002285627,0.000188474,3.85911E-06,1.98085E-08,2.5472E-11,8.18479E-15,6.55517E-19,1.27271E-23]
],
[
    [0,0.008835088,0.037781436,0.059489561,0.047240662,0.0199517,0.004444436,0.000513953,3.0467E-05,9.17791E-07,1.39665E-08],
    [0,1.116739796,1.258391634,1.019467532,0.586971899,0.211074469,0.043140468,0.004746501,0.00027281,8.05248E-06,1.20824E-07],
    [9,6.229143349,4.047400494,2.584383574,1.327756502,0.451617993,0.08967968,0.009708384,0.000552541,1.62057E-05,2.42103E-07],
    [0,1.116739796,1.258391634,1.019467532,0.586971899,0.211074469,0.043140468,0.004746501,0.00027281,8.05248E-06,1.20824E-07],
    [0,0.008835088,0.037781436,0.059489561,0.047240662,0.0199517,0.004444436,0.000513953,3.0467E-05,9.17791E-07,1.39665E-08]
],
[
    [0,0.009558137,0.045558222,0.094290178,0.126527559,0.121381463,0.082799115,0.038959005,0.012293505,0.002549088,0.000342599],
    [0,1.121515909,1.309017205,1.240218974,1.07101311,0.800335404,0.473904896,0.205472831,0.061722524,0.012406125,0.001633298],
    [9,6.238012909,4.141012088,2.989509339,2.206016695,1.504597102,0.845934298,0.356104838,0.105123357,0.02090238,0.002732481],
    [0,1.121515909,1.309017205,1.240218974,1.07101311,0.800335404,0.473904896,0.205472831,0.061722524,0.012406125,0.001633298],
    [0,0.009558137,0.045558222,0.094290178,0.126527559,0.121381463,0.082799115,0.038959005,0.012293505,0.002549088,0.000342599]
],
[
    [0,0.009616849,0.046389831,0.100088823,0.150293716,0.182445976,0.185038183,0.153990787,0.101629504,0.051494166,0.01953842],
    [0,1.121750558,1.312313706,1.263052005,1.163689526,1.035339511,0.860836208,0.632237142,0.385978935,0.186194531,0.06843535],
    [9,6.238384396,4.146214675,3.025473858,2.35155001,1.872107246,1.447815646,1.015691564,0.602687926,0.28562678,0.103799544],
    [0,1.121750558,1.312313706,1.263052005,1.163689526,1.035339511,0.860836208,0.632237142,0.385978935,0.186194531,0.06843535],
    [0,0.009616849,0.046389831,0.100088823,0.150293716,0.182445976,0.185038183,0.153990787,0.101629504,0.051494166,0.01953842]
],
[
    [0,0.00962116,0.046460652,0.100719638,0.153911673,0.196590652,0.223838776,0.230119238,0.210137529,0.165419066,0.108771727],
    [0,1.121761835,1.312524663,1.264923628,1.174386998,1.076966731,0.974333139,0.853272307,0.698261672,0.510798373,0.31994147],
    [9,6.238401627,4.146515969,3.028160041,2.366891446,1.931709979,1.610010144,1.330802943,1.046604904,0.745541773,0.458867375],
    [0,1.121761835,1.312524663,1.264923628,1.174386998,1.076966731,0.974333139,0.853272307,0.698261672,0.510798373,0.31994147],
    [0,0.00962116,0.046460652,0.100719638,0.153911673,0.196590652,0.223838776,0.230119238,0.210137529,0.165419066,0.108771727]
]
])

BIOSCREENAT_data_lineardecay = np.array([
[
    [0,0.003210825,0.006144046,0.002147448,0.000176682,3.61264E-06,1.8527E-08,2.38103E-11,7.64779E-15,6.12334E-19,1.18851E-23],
    [0,0.967975992,0.645460204,0.145416545,0.009852087,0.000182796,8.87924E-07,1.10399E-09,3.47034E-13,2.73776E-17,5.23383E-22],
    [9,5.761387432,2.46304187,0.484764988,0.031186145,0.00056499,2.70979E-06,3.34401E-09,1.04615E-12,8.22629E-17,1.56741E-21],
    [0,0.967975992,0.645460204,0.145416545,0.009852087,0.000182796,8.87924E-07,1.10399E-09,3.47034E-13,2.73776E-17,5.23383E-22],
    [0,0.003210825,0.006144046,0.002147448,0.000176682,3.61264E-06,1.8527E-08,2.38103E-11,7.64779E-15,6.12334E-19,1.18851E-23]
],
[
    [0,0.008160936,0.034422334,0.053467423,0.042032591,0.017634732,0.003911631,0.000451086,2.66903E-05,8.02973E-07,1.22078E-08],
    [0,1.072843049,1.173424199,0.927508052,0.52540863,0.187119947,0.038027923,0.004169519,0.000239115,7.04735E-06,1.05631E-07],
    [9,6.034129077,3.797240473,2.360173391,1.190841247,0.400756584,0.079091574,0.008530584,0.000484374,1.41843E-05,2.11674E-07],
    [0,1.072843049,1.173424199,0.927508052,0.52540863,0.187119947,0.038027923,0.004169519,0.000239115,7.04735E-06,1.05631E-07],
    [0,0.008160936,0.034422334,0.053467423,0.042032591,0.017634732,0.003911631,0.000451086,2.66903E-05,8.02973E-07,1.22078E-08]
],
[
    [0,0.008776909,0.041036593,0.082980008,0.108983739,0.102808922,0.069301855,0.032347164,0.010151965,0.002097215,0.000281128],
    [0,1.076922814,1.21660192,1.115282579,0.935460014,0.683537886,0.398544554,0.171067028,0.051052359,0.010216982,0.001341091],
    [9,6.041711357,3.877141017,2.705079358,1.935566813,1.288717129,0.712602203,0.296758702,0.086998622,0.017219805,0.0022441],
    [0,1.076922814,1.21660192,1.115282579,0.935460014,0.683537886,0.398544554,0.171067028,0.051052359,0.010216982,0.001341091],
    [0,0.008776909,0.041036593,0.082980008,0.108983739,0.102808922,0.069301855,0.032347164,0.010151965,0.002097215,0.000281128]
],
[
    [0,0.008823609,0.041697551,0.087582572,0.127809881,0.15104909,0.149789705,0.122531963,0.079871609,0.040118598,0.015129133],
    [0,1.077108557,1.219226146,1.133433034,1.008991881,0.869509668,0.703706004,0.506253399,0.304537551,0.145417815,0.053073872],
    [9,6.042002262,3.881283879,2.733682436,2.051096373,1.579705791,1.187561502,0.815109434,0.476183609,0.223267013,0.080543776],
    [0,1.077108557,1.219226146,1.133433034,1.008991881,0.869509668,0.703706004,0.506253399,0.304537551,0.145417815,0.053073872],
    [0,0.008823609,0.041697551,0.087582572,0.127809881,0.15104909,0.149789705,0.122531963,0.079871609,0.040118598,0.015129133]
],
[
    [0,0.00882681,0.041750084,0.088050252,0.130489001,0.161507358,0.178421385,0.178572146,0.159516946,0.123464698,0.080177468],
    [0,1.077117933,1.219382748,1.134822023,1.016921638,0.900319097,0.787548351,0.669147483,0.534020418,0.383169559,0.236618433],
    [9,6.042017973,3.881509406,2.735677213,2.062470808,1.623834876,1.307419065,1.047418608,0.802522212,0.560254196,0.339761647],
    [0,1.077117933,1.219382748,1.134822023,1.016921638,0.900319097,0.787548351,0.669147483,0.534020418,0.383169559,0.236618433],
    [0,0.00882681,0.041750084,0.088050252,0.130489001,0.161507358,0.178421385,0.178572146,0.159516946,0.123464698,0.080177468]
]
])

### Simulation with `Bioscreen`-model class of *mibitrans*

Making use of Example data of BIOSCREEN.

In [ ]:
#bio_bio; Bioscreen model with BIOSCREEN input
bio_bio_object = mbt.Bioscreen(bio_hydro, bio_att, bio_source, bio_model)

#nd; no decay
bio_bio_nd_results = bio_bio_object.run()

#dc; decay
bio_bio_object.attenuation_parameters.half_life = 0.1 * 365 #[d]
bio_bio_dc_results = bio_bio_object.run()

#inst; instant reaction
bio_bio_object.instant_reaction(bio_ea)
bio_bio_inst_results = bio_bio_object.run()

### Comparison of BIOSCREEN and `Bioscreen`-model class of *mibitrans*

Comparison for all three options of biodegradation:
* no decay
* linear decay
* instantaneous reaction


Note that BIOSCREEN  only provides limited resolution which causes visual differences in the results, particularly for instantaneous reaction model. Actually calculated values (indicated with markers) are identical.

In [ ]:
cmap = plt.get_cmap('Paired')
colors = cmap.colors[::2] 
colors_bio = cmap.colors[1::2] 

In [ ]:
for i,ti in enumerate(bio_t):
    mbt.centerline(bio_bio_nd_results, time=ti,
                   color=colors[i],lw = 4,
                   label=f"Bioscreen")
for i,ti in enumerate(bio_t):
    plt.plot(bio_x, BIOSCREEN_data_nodecay[i, 2, :],
             color=colors_bio[i],lw = 2,marker = 'o',
             linestyle="--", label=f"BIOSCREEN, t={ti}d")
    
plt.title("Comparison of BIOSCREEN and Bioscreen class without decay")
plt.legend(ncols = 2,loc = 'upper right')
plt.show()

In [ ]:
for i,ti in enumerate(bio_t):
    mbt.centerline(bio_bio_dc_results, time=ti,lw = 4,
                   color=colors[i], label=f"Bioscreen")

for i,ti in enumerate(bio_t):
    plt.plot(bio_x, BIOSCREEN_data_lineardecay[i, 2, :],
             color=colors_bio[i],lw = 2,marker = 'o',
             linestyle=":", label=f"BIOSCREEN, t={ti}d")
    
plt.title("Comparison of BIOSCREEN and Bioscreen class with linear decay")
plt.legend(ncols = 2,loc = 'upper right')
plt.show()

In [ ]:
for i,ti in enumerate(bio_t):
    mbt.centerline(bio_bio_inst_results, time=ti,
                   color=colors[i],lw = 4,
                   label=f"Bioscreen")
    
for i,ti in enumerate(bio_t):
    plt.plot(bio_x, BIOSCREEN_data_instant[i, 2, :],
             color=colors_bio[i],lw = 2,marker = 'o',
             linestyle=":", label=f"BIOSCREEN, t={ti}d")
    
plt.title("Comparison of BIOSCREEN and Bioscreen class with instant reaction")
plt.legend(ncols = 2,loc = 'upper right')
plt.show()

### Simulation with `Anatrans`-model class of *mibitrans*

Making use of Example data of BIOSCREEN.

In [ ]:
#bio_ana; Anatrans model with BIOSCREEN input
bio_ana_object = mbt.Anatrans(bio_hydro, bio_att, bio_source, bio_model)
bio_ana_nd_results = bio_ana_object.run()

bio_ana_object.attenuation_parameters.half_life = 0.1 * 365 #[d]
bio_ana_dc_results = bio_ana_object.run()

bio_ana_object.instant_reaction(bio_ea)
bio_ana_inst_results = bio_ana_object.run()

### Comparison of BIOSCREEN and `Anatrans`-model class of *mibitrans*

Comparison for all three options of biodegradation:
* no decay
* linear decay
* instantaneous reaction

Note that BIOSCREEN only provides limited resolution (indicated with markers). Differences in results, particularly for instantaneous reactin model are caused by that.

In [ ]:
time_bio = bio_t
for i,ti in enumerate(bio_t):
    mbt.centerline(bio_ana_nd_results, time=ti,
                   color=colors[i],lw = 4,
                   label=f"Anatrans, t={ti}d")
for i,ti in enumerate(bio_t):
    plt.plot(bio_x, BIOSCREEN_data_nodecay[i, 2, :],
             color=colors_bio[i],lw = 2,marker = 's',
             linestyle="--", label=f"BIOSCREEN, t={ti}d")
    
plt.title("Comparison of BIOSCREEN and Anatrans models without decay")
plt.legend(ncols=2,loc = 'upper right')
plt.show()

In [ ]:
for i,ti in enumerate(bio_t):
    mbt.centerline(bio_ana_dc_results, time=ti,
                   color=colors[i],lw = 4,
                   label=f"Anatrans")
    
for i,ti in enumerate(bio_t):
    plt.plot(bio_x, BIOSCREEN_data_lineardecay[i, 2, :],
             color=colors_bio[i],lw = 2,marker = 's',
             linestyle="--", label=f"BIOSCREEN, t={ti}d")
    
plt.title("Comparison of BIOSCREEN and Anatrans models with linear decay")
plt.legend(ncols =2,loc = 'upper right')
plt.show()

In [ ]:
for i,ti in enumerate(bio_t):
    mbt.centerline(bio_ana_inst_results, time=ti,
                   color=colors[i],lw = 4,
                   label=f"Anatrans, t={ti}")
    
for i,ti in enumerate(bio_t):
    plt.plot(bio_x, BIOSCREEN_data_instant[i, 2, :],
             color=colors_bio[i],lw = 2,marker = 's',
             linestyle="--", label=f"BIOSCREEN, t={ti}d")
    
plt.title("Comparison of BIOSCREEN and Anatrans models with instant reaction")
plt.legend(ncols =2,loc = 'upper right')
plt.show()

### Simulation with `Mibitrans`-model class of *mibitrans*

Making use of Example data of BIOSCREEN-AT.

As the exact solution uses an integral, run time is longer than that of the Bioscreen solution, depending on model resolution. Using verbose when running the model is recommended, to keep track of the calculation process.

In [ ]:
# bat_mbt; BIOSCREEN-AT data with Mibitrans model
bat_mbt_object = mbt.Mibitrans(
    hydrological_parameters=bat_hydro,
    attenuation_parameters=bat_att,
    source_parameters=bat_source,
    model_parameters=bat_model,
)
bat_mbt_nd_results = bat_mbt_object.run()

bat_mbt_object.attenuation_parameters.half_life = 365 * 10 #[d]
bat_mbt_dc_results = bat_mbt_object.run()

bat_mbt_object.instant_reaction(electron_acceptors=bat_ea)
bat_mbt_inst_results = bat_mbt_object.run()

### Comparison of BIOSCREEN-AT and `Mibitrans`-model class of *mibitrans*

Comparison for two options of biodegradation:
* no decay
* linear decay

Note that BIOSCREEN-AT does not have an implementation of the instantaneous reaction model for degradation (despite BIOSCREEN-AT taking in instant reaction parameters). Consequently, there is no comparison. 

BIOSCREEN-AT (just as BIOSCREEN) only provides limited resolution  which causes visual differences in the results. Actually calculated values (indicated with markers) of BIOSCREEN-AT and the `Mibitrans`-model class of *mibitrans* are identical. 

In [ ]:
for i,ti in enumerate(bat_t):
    mbt.centerline(bat_mbt_nd_results, time=ti,
                   color=colors[i],lw = 4,
                   label=f"Mibitrans")
for i,ti in enumerate(bat_t):
    plt.plot(bat_x, BIOSCREENAT_data_nodecay[i, 2, :],
             color=colors_bio[i],lw = 2,marker = '*',
             linestyle="--", label=f"BIOSCREEN-AT, t={ti}")
    
plt.title("Comparison of BIOSCREEN-AT and Mibitrans models with no decay")
plt.legend(ncols =2,loc = 'upper right')
plt.show()

In [ ]:
bat_mbt_dc_results.plume_3d()

In [ ]:
for i,ti in enumerate(bat_t):
    mbt.centerline(bat_mbt_dc_results, time=ti,
                   color=colors[i],lw = 4,
                   label=f"Mibitrans")
for i,ti in enumerate(bat_t):
    plt.plot(bat_x, BIOSCREENAT_data_lineardecay[i, 2, :],
             color=colors_bio[i],lw = 2,marker = '*',
             ls="--", label=f"BIOSCREEN-AT, t={ti}")

plt.title("Comparison of BIOSCREEN-AT and Mibitrans models with linear decay")
plt.show()

### Quantify differences



In [ ]:
###  Functions for quantitative comparison
import differences as df

#### Redefine model dimension for *mibitrans* models to match data from BIOSCREEN and BIOSCREEN-AT

In [ ]:
bio_model_comp = mbt.ModelParameters(
    model_length=1450 / ft, #[m]
    model_width=300 / ft, #[m]
    model_time=5 * 365, #[d]
    dx=145 / ft, #[m]
    dy=75 / ft, #[m]
    dt=365/2, #[d]
)

bat_model_comp = mbt.ModelParameters(
    model_length=1500 / ft, #[m]
    model_width=300 / ft, #[m]
    model_time=5 * 365, #[d]
    dx=150 / ft, #[m]
    dy=75 / ft, #[m]
    dt=365, #[d]
)

In [ ]:
bio_object_comp = mbt.Bioscreen(bio_hydro, bio_att, bio_source, bio_model_comp)
bio_results_comp_nodecay = bio_object_comp.run()

bio_object_comp.attenuation_parameters.half_life = 0.1 * 365
bio_results_comp_linear= bio_object_comp.run()

bio_object_comp.instant_reaction(bio_ea)
bio_results_comp_instant = bio_object_comp.run()

mbt_object_comp = mbt.Mibitrans(bat_hydro, bat_att, bat_source, bat_model_comp)
mbt_results_comp_nodecay = mbt_object_comp.run()

mbt_object_comp.attenuation_parameters.half_life = 10*365
mbt_results_comp_linear = mbt_object_comp.run()

In [ ]:
bio_time_index = [0,1,3,-1]

In [ ]:
#Quick check if model dimensions are similar to those used to obtain BIOSCREEN results
print(np.sum(abs(bio_x - bio_results_comp_instant.x)))
print(np.sum(abs(bio_y - bio_results_comp_instant.y)))
print(np.sum(abs(bio_t - bio_results_comp_instant.t[bio_time_index])))
print(np.sum(abs(bat_x - mbt_results_comp_linear.x)))
print(np.sum(abs(bat_y - mbt_results_comp_linear.y)))
print(np.sum(abs(bat_t - mbt_results_comp_linear.t)))

In [ ]:
max_source_conc = bio_source.source_zone_concentration[0]

mad_bio_bio_nd = df.mean_absolute_difference(BIOSCREEN_data_nodecay / max_source_conc, bio_results_comp_nodecay.relative_cxyt[bio_time_index,:,:])

mad_bio_bio_lindec = df.mean_absolute_difference(BIOSCREEN_data_lineardecay / max_source_conc, bio_results_comp_linear.relative_cxyt[bio_time_index,:,:])

mad_bio_bio_instant = df.mean_absolute_difference(BIOSCREEN_data_instant / max_source_conc, bio_results_comp_instant.relative_cxyt[bio_time_index,:,:])

In [ ]:
print("Mean absolute difference in relative concentration between BIOSCREEN and Bioscreen model class with no decay:",mad_bio_bio_nd)
print("Mean absolute difference in relative concentration between BIOSCREEN and Bioscreen model class with linear decay:",mad_bio_bio_lindec)
print("Mean absolute difference in relative concentration between BIOSCREEN and Bioscreen model class with instant reaction:",mad_bio_bio_instant)

In [ ]:
max_source_conc = bat_source.source_zone_concentration[0]

mad_bat_mibi_nd = df.mean_absolute_difference(BIOSCREENAT_data_nodecay / max_source_conc, mbt_results_comp_nodecay.relative_cxyt)

mad_bat_mibi_lindec = df.mean_absolute_difference(BIOSCREENAT_data_lineardecay / max_source_conc, mbt_results_comp_linear.relative_cxyt)

In [ ]:
print("Mean absolute difference in relative concentration between BIOSCREEN-AT and Mibitrans model class with no decay:",mad_bat_mibi_nd)
print("Mean absolute difference in relative concentration between BIOSCREEN-AT and Mibitrans model class with linear decay:",mad_bat_mibi_lindec)